<a href="https://colab.research.google.com/github/cysorianoc/Hugging_Face/blob/main/Notebook_4_L5_Zero_Shot_Audio_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5: Zero-Shot Audio Classification


Audio classification has many applications, for instance identify a spoken language, identify a type of bird and so on.

Here we will build a sound classifier.

In a traditional classificator, a model predicts a label from a predefined set of classes it was trained on. If there are no models trained on your specific set of classes you would have to collect a dataset and fine-tune a model.

Here we have an alternative approach that does not require fine-tuning.


##


- In the classroom, the libraries have already been installed for you.
- If you are running this code on your own machine, please install the following:
```
    !pip install transformers
    !pip install datasets
    !pip install soundfile
    !pip install librosa
```

The `librosa` library may need to have [ffmpeg](https://www.ffmpeg.org/download.html) installed.
- This page on [librosa](https://pypi.org/project/librosa/) provides installation instructions for ffmpeg.

- Here is some code that suppresses warning messages.

In [1]:
from transformers.utils import logging
logging.set_verbosity_error()

### Prepare the dataset of audio recordings

In [3]:
from datasets import load_dataset, load_from_disk

# This dataset is a collection of different sounds of 5 seconds
dataset = load_dataset("ashraq/esc50",
                      split="train[0:10]")
# dataset = load_from_disk("./models/ashraq/esc50/train")

# This dataset is a labeled collection of five-second environmental sounds,
# such as sounds made by animals and humans, nature sounds, urban sounds ...

README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

data/train-00000-of-00002-2f1ab7b824ec75(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00000-of-00002-2f1ab7b824ec75(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-27425e5c1846b4(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00001-of-00002-27425e5c1846b4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
# Let's see an example
audio_sample = dataset[0]

In [7]:
audio_sample

{'filename': '1-100032-A-0.wav',
 'fold': 1,
 'target': 0,
 'category': 'dog',
 'esc10': True,
 'src_file': 100032,
 'take': 'A',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7db0b023e570>}

In [8]:
from IPython.display import Audio as IPythonAudio
IPythonAudio(audio_sample["audio"]["array"],
             rate=audio_sample["audio"]["sampling_rate"])

### Build the `audio classification` pipeline using 🤗 Transformers Library

In [9]:
from transformers import pipeline

For this type od classificaion we may need a pre-trained CLAP model.

To classify the audio, you only need the array of audio data, however there is a required sampling rate.

A sound wave contains an infinite number of signal values in a given time. But an audio in your computer is a series of discrete values, known as digital representation.

To get a digital representation, we capture the sound with a microphone. Then the analog signal is converted into an electrical signal. Then, the electrical signal is samples to get the digital representation.

## Sampling rate

Sampling means measuring the value of a continuous signal at fixed time steps. A very important characteristic of an audio signal is the sampling rate, it is the number of samples taken in one second and it is measured in hertz or kilohertz.

Examples:
- 8000 Hz: telephone, walkie-talkie
- 16000 Hz: human speech recording
- 192000 Hz: high-resolution audio


Why sampling rate is important when working with AI models ?


A five-second sound at a sampling rate of 8 kilohertz will be represented as a series of 40,000 signal values.

The same five-second sound sampled at 16 kilohertz will be represented as a series of 80,000 signal values.


And at 192 kilohertz, it will be represented with almost a million values


For a transformer model, these three arrays are very different. Transformer models treat input as sequences and rely on attention mechanisms to learn audio representation. They are trained on datasets where all examples have the same sampling rate and they do not generalize well to other sampling rates.

So for a transformer trained on 16 kHz audio, an array of 960000 values will look like a 60-second recording at 16 kHz (60*16000=960000).


If a transformer model has been trained with audio samples, each recorded at a sampling rate of 16 kilohertz, it's going to view any input as if it was recorded at the same sampling rate.


In [14]:
zero_shot_classifier = pipeline(
    task="zero-shot-audio-classification",
    model="laion/clap-htsat-unfused")

config.json:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  615MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  614MB            

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

More info on [laion/clap-htsat-unfused](https://huggingface.co/laion/clap-htsat-unfused).

### Sampling Rate for Transformer Models
- How long does 1 second of high resolution audio (192,000 Hz) appear to the Whisper model (which is trained to expect audio files at 16,000 Hz)?

In [11]:
(1 * 192000) / 16000
# so in this model, the recording at 16000 Hz will look like 12 seconds

12.0

- The 1 second of high resolution audio appears to the model as if it is 12 seconds of audio.

- How about 5 seconds of audio?

In [ ]:
(5 * 192000) / 16000

60.0

- 5 seconds of high resolution audio appears to the model as if it is 60 seconds of audio.

The original audio was 5 seconds but with a lot of samples per second. For a model trained at a lower sampling rate this audio will look longer.

In [15]:
# Let's check the sampling rate at which the model was trained
zero_shot_classifier.feature_extractor.sampling_rate

48000

In [16]:
# What about the sampling rate of the audio example ?
audio_sample["audio"]["sampling_rate"]

44100

In this case the sampling rates are similar, and the model may do ok, but this may not be the case for other audios.

* Set the correct sampling rate for the input and the model.

In [17]:
from datasets import Audio

In [18]:
# We can cast the dataset into the appropriate sampling rate as follows
dataset = dataset.cast_column(
    "audio",
     Audio(sampling_rate=48_000))

In [19]:
audio_sample = dataset[0]

In [20]:
audio_sample

{'filename': '1-100032-A-0.wav',
 'fold': 1,
 'target': 0,
 'category': 'dog',
 'esc10': True,
 'src_file': 100032,
 'take': 'A',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7db0a23287d0>}

In [21]:
audio_sample["audio"]["sampling_rate"]

48000

In [22]:
# We need to provide the pipeline with the candidate labels to compute similarity scores
candidate_labels = ["Sound of a dog",
                    "Sound of vacuum cleaner"]

In [23]:
# We pass the candidate labels and the audio into the pipeline
zero_shot_classifier(audio_sample["audio"]["array"],
                     candidate_labels=candidate_labels)

[{'score': 0.9996943473815918, 'label': 'Sound of a dog'},
 {'score': 0.0003057112917304039, 'label': 'Sound of vacuum cleaner'}]

In [25]:
# We can try other labels, unrelated with a dog barking
candidate_labels = ["Sound of a child crying",
                    "Sound of vacuum cleaner",
                    "Sound of a bird singing",
                    "Sound of an airplane"]

In [26]:
zero_shot_classifier(audio_sample["audio"]["array"],
                     candidate_labels=candidate_labels)

[{'score': 0.4876127541065216, 'label': 'Sound of a bird singing'},
 {'score': 0.2569237947463989, 'label': 'Sound of vacuum cleaner'},
 {'score': 0.24024361371994019, 'label': 'Sound of an airplane'},
 {'score': 0.01521978061646223, 'label': 'Sound of a child crying'}]

### Try it yourself!
- Try this model with some other labels and audio files!